In [1]:
from guardrails import Guard
from langchain_chroma import Chroma
from pathlib import Path
from dotenv import load_dotenv
from config.parameter_config import params_config
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from guardrails_ai.provenance_embeddings import ProvenanceEmbeddings
# from guardrails_ai.reading_time import ReadingTime
from guardrails_ai.relevancy_evaluator import RelevancyEvaluator
import numpy as np

/Users/himanshuarora/llmops/llmops-rag-app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
app_params = params_config.rag_app

In [4]:
REPO_ROOT = Path.cwd().parent
VECTOR_STORE_DIR = REPO_ROOT / "saved-embeddings"

In [5]:
embedder = OpenAIEmbeddings(model=app_params.embedding_model,
                            dimensions=app_params.embedding_dimensions) # 1536

In [6]:
chunk_size = app_params.chunk_size
chunk_overlap = app_params.chunk_overlap

In [7]:
def load_knowledge_base():
    # vector store
    vs = Chroma(collection_name=app_params.collection_name,
            embedding_function=embedder,
            persist_directory=VECTOR_STORE_DIR.as_posix())


    return vs

In [8]:
if VECTOR_STORE_DIR.exists():
    vs = load_knowledge_base()

In [9]:
def get_retriever():
    # create the retriever
    retriever = vs.as_retriever(search_type=app_params.search_type,
                                search_kwargs={"k":app_params.k})
    

    if app_params.contextual_compression:
        # compressor
        compression_llm = ChatOpenAI(model=app_params.compression_llm)
        compressor = LLMChainExtractor.from_llm(compression_llm)
        
        # compression retriever
        compression_retriever = ContextualCompressionRetriever(
            base_compressor=compressor,
            base_retriever=retriever
        )
        return compression_retriever
    
    return retriever

retriever = get_retriever()

In [11]:
def embed_func(texts):
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

    if isinstance(texts, str):
        return np.array(embedding_model.embed_query(texts))

    return np.array(embedding_model.embed_documents(texts))

In [27]:
# generate the guardrails

# reading_time = ReadingTime(
#     reading_time=3,
#     on_fail="noop"
# )

relevancy_eval = RelevancyEvaluator(
    llm_callable="gpt-5-mini",
    on_fail="refrain"
)

hallucination_check = ProvenanceEmbeddings(
    threshold=0.7,
    validation_method="sentence",
    on_fail="refrain",
)

In [28]:
# create the guard

guard = Guard().use(
    # reading_time,
    relevancy_eval,
    hallucination_check
)

In [14]:
query = "What is the diffence between online vs offline evals"

In [15]:
retriever_output = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in retriever_output])

In [16]:
llm = ChatOpenAI(model=app_params.llm)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Answer the user query
                  based on the given context only. If you do not know the answer
                  say I don't know. Do not add any preamble to the response"""),
    ("human", "context: {context}\n\nquery: {query}")
])

rag_chain = prompt | llm | StrOutputParser()
response = rag_chain.invoke({"context": context, "query": query})

In [21]:
context

"Online eval is evaluating your system,\nOn live production traffic,\nAfter deployment,\nAs real users interact with it.\nThe simplest definition of online eval is,\nThat it is a different type of evaluation,\nWhich we run on production,\nAfter our software is deployed.\nAnd the biggest characteristic of it is,\nThat it works without an answer.\nIt works without a golden data set.\nThis is the biggest feature of online eval.\nAnd that is why it is super critical,\nBecause it helps us,\nThat our software, which is deployed,\nKeeps running properly.\nIt tells us that,\nThere is no problem with online.\nIn that sense, online evals are super important.\nSo, on the basis of the discussion till now,\nIf I quickly lay out a difference,\nBetween offline and online,\nThere are 4-5 pointers.\nFirst of all, offline eval,\nIs before deployment,\nAfter online eval deployment.\nTalking about data,\nIn offline eval, you have a fixed data set,\nWhich you create,\nGolden data set.\nHere, you don't have

In [18]:
print(response)

- Timing:
  - Offline eval: before deployment (in development/CI).
  - Online eval: after deployment, on live production traffic.

- Data / labels:
  - Offline: uses a fixed, created golden dataset with answers.
  - Online: runs without a golden dataset or ground-truth answers; uses real user traffic.

- Purpose:
  - Offline: checks whether the application/model is working correctly (functional correctness, comparisons, CI).
  - Online: detects drift and verifies the application is running normally in production (real-world performance).

- Scale, cost and speed:
  - Offline: fast, cheap, repeatable (small curated dataset).
  - Online: can be large-scale and costly to monitor every interaction; often uses sampling to reduce cost.

- Characteristic features:
  - Offline: suitable for controlled evaluation, version comparison.
  - Online: critical for monitoring real-world behavior and detecting issues without labeled data.

- Relationship:
  - They are complementary, not replacements; b

In [32]:
context = """Offline evaluation happens before deployment, using a fixed golden dataset that you
create ahead of time. It's fast, cheap, and repeatable. Online evaluation happens
after deployment, on live production traffic, without a golden dataset."""


query = "What is the difference between offline and online evals?"

response = """MLflow was created by Databricks in 2018 and its enterprise tier costs $499/month,
which includes unlimited experiment tracking and AI copilot features."""

In [ ]:
validation_metadata = {
    "sources": [context],
    "original_prompt": query,
    "embed_function": embed_func,
    "chunk_size": 2,
    "chunk_overlap": 1
}

In [34]:
try:
    outcome = guard.validate(response, metadata=validation_metadata)
    if outcome.validated_output is None:
        print("Please ask again. No relevant response generated")
    if outcome.validation_passed:
        print(f"Response: {outcome.validated_output}")
except Exception as e:
    print(e) 

Please ask again. No relevant response generated
